<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/dataPreProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#═══════════════════════════════════════════════════════
# STEP 1: Upload your 3 original research datasets
#═══════════════════════════════════════════════════════
from google.colab import files
import pandas as pd

print("Upload these 3 original datasets:")
print("   1. PROMISE-relabeled-NICE.csv")
print("   2. dcai24_src_dataset.xlsx")
print("   3. issues.csv")
print("")
print("Click Choose Files below...")

uploaded = files.upload()

print("\n✅ Files uploaded:")
for f in uploaded.keys():
    print(f"   → {f}")

Upload these 3 original datasets:
   1. PROMISE-relabeled-NICE.csv
   2. dcai24_src_dataset.xlsx
   3. issues.csv

Click Choose Files below...


Saving PROMISE-relabeled-NICE.csv to PROMISE-relabeled-NICE.csv
Saving dcai24_src_dataset.xlsx to dcai24_src_dataset.xlsx
Saving issues.csv to issues.csv

✅ Files uploaded:
   → PROMISE-relabeled-NICE.csv
   → dcai24_src_dataset.xlsx
   → issues.csv


In [3]:
#═══════════════════════════════════════════════════════
# STEP 2: Load and inspect all 3 datasets
#═══════════════════════════════════════════════════════
import pandas as pd

# Load all 3 datasets
df_promise = pd.read_csv("PROMISE-relabeled-NICE.csv")
df_dcai    = pd.read_excel("dcai24_src_dataset.xlsx")
df_issues  = pd.read_csv("issues.csv")

print("=" * 55)
print("   DATASET OVERVIEW")
print("=" * 55)

print(f"\n1. PROMISE-relabeled-NICE.csv")
print(f"   Rows    : {len(df_promise):,}")
print(f"   Columns : {list(df_promise.columns)}")

print(f"\n2. dcai24_src_dataset.xlsx")
print(f"   Rows    : {len(df_dcai):,}")
print(f"   Columns : {list(df_dcai.columns)}")

print(f"\n3. issues.csv")
print(f"   Rows    : {len(df_issues):,}")
print(f"   Columns : {list(df_issues.columns)}")

   DATASET OVERVIEW

1. PROMISE-relabeled-NICE.csv
   Rows    : 622
   Columns : ['ProjectID', 'RequirementText', 'IsFunctional', 'IsQuality', 'Availability (A)', 'Fault Tolerance (FT)', 'Legal (L)', 'Look & Feel (LF)', 'Maintainability (MN)', 'Operability (O)', 'Performance (PE)', 'Portability (PO)', 'Scalability (SC)', 'Security (SE)', 'Usability (US)', 'Other (OT)']

2. dcai24_src_dataset.xlsx
   Rows    : 3,482
   Columns : ['Requirement', 'Type', 'Specific_Type', 'Security_Category', 'Dataset_Name']

3. issues.csv
   Rows    : 20,479
   Columns : ['idproject', 'issuekey', 'created', 'title', 'description', 'storypoints']


In [4]:
#═══════════════════════════════════════════════════════
# STEP 3: Process PROMISE dataset
# Map requirement type → Priority label
# Reference: Cleland-Huang et al. PROMISE Repository
#═══════════════════════════════════════════════════════

# PROMISE has binary columns for each type
# We map type → priority based on Agile importance

def get_promise_priority(row):
    # Security → High (system safety critical)
    if row.get('Security (SE)', 0) == 1:
        return 'High', 0
    # Fault Tolerance → High (system reliability)
    if row.get('Fault Tolerance (FT)', 0) == 1:
        return 'High', 0
    # Availability → High (system must be accessible)
    if row.get('Availability (A)', 0) == 1:
        return 'High', 0
    # Performance → Medium (important but not critical)
    if row.get('Performance (PE)', 0) == 1:
        return 'Medium', 1
    # Scalability → Medium
    if row.get('Scalability (SC)', 0) == 1:
        return 'Medium', 1
    # Operability → Medium
    if row.get('Operability (O)', 0) == 1:
        return 'Medium', 1
    # Functional → Medium (core features)
    if row.get('IsFunctional', 0) == 1:
        return 'Medium', 1
    # Usability → Low (nice to have)
    if row.get('Usability (US)', 0) == 1:
        return 'Low', 2
    # Maintainability → Low (future work)
    if row.get('Maintainability (MN)', 0) == 1:
        return 'Low', 2
    # Portability → Low
    if row.get('Portability (PO)', 0) == 1:
        return 'Low', 2
    # Look & Feel → Low (cosmetic)
    if row.get('Look & Feel (LF)', 0) == 1:
        return 'Low', 2
    # Legal → Low
    if row.get('Legal (L)', 0) == 1:
        return 'Low', 2
    # Default
    return 'Medium', 1

# Apply priority mapping
df_promise['priority'], df_promise['priority_id'] = zip(
    *df_promise.apply(get_promise_priority, axis=1)
)

# Keep only text and labels
df_promise_clean = df_promise[
    ['RequirementText', 'priority', 'priority_id']
].copy()
df_promise_clean = df_promise_clean.rename(
    columns={'RequirementText': 'text'}
)
df_promise_clean['source'] = 'PROMISE'
df_promise_clean = df_promise_clean.dropna(subset=['text'])
df_promise_clean = df_promise_clean[
    df_promise_clean['text'].str.len() > 5
]

print("=" * 55)
print("   PROMISE DATASET — PROCESSED")
print("=" * 55)
print(f"   Total rows : {len(df_promise_clean):,}")
print(f"\n   Priority Distribution:")
print(df_promise_clean['priority'].value_counts())
print(f"\n   Sample rows:")
print(df_promise_clean[['text','priority']].head(5).to_string())

   PROMISE DATASET — PROCESSED
   Total rows : 622

   Priority Distribution:
priority
Medium    396
High      119
Low       107
Name: count, dtype: int64

   Sample rows:
                                                                                                                                                                                                          text priority
0                                                                                                                                                     'The system shall refresh the display every 60 seconds.'   Medium
1                                                                                                           'The application shall match the color of the schema set forth by Department of Homeland Security'      Low
2                                               'If projected the data must be readable. On a 10x10 projection screen 90% of viewers must be able to read Event / Activity data from

In [5]:
#═══════════════════════════════════════════════════════
# STEP 4: Process dcai24 dataset
# Map Specific_Type → Priority label
# Reference: dcai24 Software Requirements Dataset
#═══════════════════════════════════════════════════════

# dcai24 type codes → Priority mapping
dcai_type_to_priority = {
    # HIGH — Security and critical reliability
    'SE'     : ('High',   0),   # Security
    'FT'     : ('High',   0),   # Fault Tolerance
    'A'      : ('High',   0),   # Availability

    # MEDIUM — Important functional requirements
    'F'      : ('Medium', 1),   # Functional
    'PE'     : ('Medium', 1),   # Performance
    'SC'     : ('Medium', 1),   # Scalability
    'O'      : ('Medium', 1),   # Operability

    # LOW — Nice to have
    'US'     : ('Low',    2),   # Usability
    'MN'     : ('Low',    2),   # Maintainability
    'PO'     : ('Low',    2),   # Portability
    'LF'     : ('Low',    2),   # Look & Feel
    'L'      : ('Low',    2),   # Legal

    # DEFAULT
    'NONSE'  : ('Medium', 1),
    'NONSEP0': ('Low',    2),
}

def get_dcai_priority(specific_type):
    return dcai_type_to_priority.get(
        str(specific_type).strip(),
        ('Medium', 1)   # default
    )

df_dcai['priority'], df_dcai['priority_id'] = zip(
    *df_dcai['Specific_Type'].apply(get_dcai_priority)
)

df_dcai_clean = df_dcai[
    ['Requirement', 'priority', 'priority_id']
].copy()
df_dcai_clean = df_dcai_clean.rename(
    columns={'Requirement': 'text'}
)
df_dcai_clean['source'] = 'dcai24'
df_dcai_clean = df_dcai_clean.dropna(subset=['text'])
df_dcai_clean = df_dcai_clean[
    df_dcai_clean['text'].str.len() > 5
]

print("=" * 55)
print("   DCAI24 DATASET — PROCESSED")
print("=" * 55)
print(f"   Total rows : {len(df_dcai_clean):,}")
print(f"\n   Priority Distribution:")
print(df_dcai_clean['priority'].value_counts())
print(f"\n   Sample rows:")
print(df_dcai_clean[['text','priority']].head(5).to_string())

   DCAI24 DATASET — PROCESSED
   Total rows : 3,471

   Priority Distribution:
priority
High      1590
Medium    1558
Low        323
Name: count, dtype: int64

   Sample rows:
                                                                                                                                                                    text priority
0                                                                                          The system must authenticate users prior to accessing an application or data.     High
1                                                                                 The system must prevent access to applications or data to all non-authenticated users.     High
2                                                                                         The system should provide the ability to implement a Chain of Trust agreement.     High
3  The system must authenticate users using at least one of the following authentication mechanisms: username/pa

In [6]:
#═══════════════════════════════════════════════════════
# STEP 5: Process issues.csv dataset
# Map Story Points → Priority label
# Reference: Jira Issues Dataset with Story Points
#
# Rationale:
# Story Points measure EFFORT not priority directly
# However in Agile, low-effort high-value items
# (small story points) are done first
# This is consistent with "quick wins" strategy
#═══════════════════════════════════════════════════════

# Remove rows with missing values
df_issues_clean = df_issues.dropna(
    subset=['storypoints', 'title']
).copy()
df_issues_clean = df_issues_clean[
    df_issues_clean['title'].str.len() > 5
]

# Story Points → Priority mapping
# Low story points = small task = do first (High priority)
# High story points = complex task = do later (Low priority)
def storypoints_to_priority(sp):
    if sp <= 2:    return 'High',   0   # Quick wins — do first
    elif sp <= 5:  return 'Medium', 1   # Medium effort
    else:          return 'Low',    2   # Complex — do later

df_issues_clean['priority'], df_issues_clean['priority_id'] = zip(
    *df_issues_clean['storypoints'].apply(storypoints_to_priority)
)

df_issues_final = df_issues_clean[
    ['title', 'priority', 'priority_id']
].copy()
df_issues_final = df_issues_final.rename(
    columns={'title': 'text'}
)
df_issues_final['source'] = 'issues'

print("=" * 55)
print("   ISSUES DATASET — PROCESSED")
print("=" * 55)
print(f"   Total rows : {len(df_issues_final):,}")
print(f"\n   Story Points → Priority Mapping:")
print(f"   1-2  points  → High   (quick wins)")
print(f"   3-5  points  → Medium (moderate effort)")
print(f"   6+   points  → Low    (complex tasks)")
print(f"\n   Priority Distribution:")
print(df_issues_final['priority'].value_counts())
print(f"\n   Sample rows:")
print(df_issues_final[['text','priority']].head(5).to_string())

   ISSUES DATASET — PROCESSED
   Total rows : 20,472

   Story Points → Priority Mapping:
   1-2  points  → High   (quick wins)
   3-5  points  → Medium (moderate effort)
   6+   points  → Low    (complex tasks)

   Priority Distribution:
priority
High      9882
Medium    5319
Low       5271
Name: count, dtype: int64

   Sample rows:
                                                               text priority
0       (feat): change 'from' email address to 'no-reply@minds.com'     High
1         (bug): Link previews disappear when editing a status post   Medium
2               (bug): subscribed blog post duplicates filling feed     High
3            (bug): Cant delete a reply to a comment on a blog post   Medium
4  Counter in group convo-feeds counts 2 seconds for every 1 second   Medium


In [7]:
#═══════════════════════════════════════════════════════
# STEP 6: Combine all 3 processed datasets
#═══════════════════════════════════════════════════════
import pandas as pd

df_combined = pd.concat(
    [df_promise_clean, df_dcai_clean, df_issues_final],
    ignore_index=True
)

# Clean combined dataset
df_combined = df_combined.dropna(subset=['text', 'priority'])
df_combined = df_combined[df_combined['text'].str.len() > 5]
df_combined = df_combined.drop_duplicates(subset=['text'])
df_combined = df_combined.reset_index(drop=True)

print("=" * 55)
print("   COMBINED DATASET SUMMARY")
print("=" * 55)
print(f"\n   Source Breakdown:")
print(f"   PROMISE : {len(df_promise_clean):>6,} rows")
print(f"   dcai24  : {len(df_dcai_clean):>6,} rows")
print(f"   issues  : {len(df_issues_final):>6,} rows")
print(f"   ──────────────────────")
print(f"   Total   : {len(df_combined):>6,} rows")
print(f"\n   Priority Distribution:")
print(df_combined['priority'].value_counts())
print(f"\n   Columns: {list(df_combined.columns)}")

   COMBINED DATASET SUMMARY

   Source Breakdown:
   PROMISE :    622 rows
   dcai24  :  3,471 rows
   issues  : 20,472 rows
   ──────────────────────
   Total   : 23,543 rows

   Priority Distribution:
priority
High      11195
Medium     6792
Low        5556
Name: count, dtype: int64

   Columns: ['text', 'priority', 'priority_id', 'source']


In [8]:
#═══════════════════════════════════════════════════════
# STEP 7: Split into Train / Validation / Test sets
# Stratified split to maintain class balance
#═══════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

# 80% Train — 20% remaining
train_df, temp_df = train_test_split(
    df_combined,
    test_size  = 0.2,
    stratify   = df_combined['priority_id'],
    random_state = 42
)

# 10% Validation — 10% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size  = 0.5,
    stratify   = temp_df['priority_id'],
    random_state = 42
)

print("=" * 55)
print("   DATASET SPLIT SUMMARY")
print("=" * 55)
print(f"\n   Train set : {len(train_df):,} rows (80%)")
print(f"   Val set   : {len(val_df):,} rows (10%)")
print(f"   Test set  : {len(test_df):,} rows (10%)")

print(f"\n   Train Priority Balance:")
print(train_df['priority'].value_counts())

print(f"\n   Val Priority Balance:")
print(val_df['priority'].value_counts())

print(f"\n   Test Priority Balance:")
print(test_df['priority'].value_counts())

   DATASET SPLIT SUMMARY

   Train set : 18,834 rows (80%)
   Val set   : 2,354 rows (10%)
   Test set  : 2,355 rows (10%)

   Train Priority Balance:
priority
High      8956
Medium    5433
Low       4445
Name: count, dtype: int64

   Val Priority Balance:
priority
High      1119
Medium     679
Low        556
Name: count, dtype: int64

   Test Priority Balance:
priority
High      1120
Medium     680
Low        555
Name: count, dtype: int64


In [9]:
#═══════════════════════════════════════════════════════
# STEP 8: Save all 4 datasets as CSV files
#═══════════════════════════════════════════════════════

df_combined.to_csv("combined_labeled_dataset.csv", index=False)
train_df.to_csv(   "train_combined.csv",           index=False)
val_df.to_csv(     "val_combined.csv",             index=False)
test_df.to_csv(    "test_combined.csv",            index=False)

print("=" * 55)
print("   ALL FILES SAVED SUCCESSFULLY")
print("=" * 55)
print(f"\n   combined_labeled_dataset.csv → {len(df_combined):,} rows")
print(f"   train_combined.csv           → {len(train_df):,} rows")
print(f"   val_combined.csv             → {len(val_df):,} rows")
print(f"   test_combined.csv            → {len(test_df):,} rows")

   ALL FILES SAVED SUCCESSFULLY

   combined_labeled_dataset.csv → 23,543 rows
   train_combined.csv           → 18,834 rows
   val_combined.csv             → 2,354 rows
   test_combined.csv            → 2,355 rows


In [10]:
#═══════════════════════════════════════════════════════
# STEP 9: Download all 4 files to your computer
#═══════════════════════════════════════════════════════
from google.colab import files

print("Downloading files...")

files.download("combined_labeled_dataset.csv")
files.download("train_combined.csv")
files.download("val_combined.csv")
files.download("test_combined.csv")

print("\n✅ All 4 files downloaded!")
print("\n   You can now show your supervisor:")
print("   → combined_labeled_dataset.csv  (full dataset)")
print("   → train_combined.csv            (80% for training)")
print("   → val_combined.csv              (10% for validation)")
print("   → test_combined.csv             (10% for testing)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All 4 files downloaded!

   You can now show your supervisor:
   → combined_labeled_dataset.csv  (full dataset)
   → train_combined.csv            (80% for training)
   → val_combined.csv              (10% for validation)
   → test_combined.csv             (10% for testing)
